# Script de python sencillo para descargar datos de EAGLE 

#### Antes de empezar: una herramienta útil para descargar datos de EAGLE desde Python, es el paquete eagleSQLtools.

#### Repositorio del paquete: https://github.com/kyleaoman/eagleSqlTools

#### Para instalarlo, en una terminal correr el siguiente comando:

#### pip install eaglesqltools

#### Importar paquetes necesarios. Agregar paquete si hace falta

In [23]:
import matplotlib.pyplot as plt
import eagleSqlTools as sql
from astropy.io import ascii
from astropy.table import Table
import numpy as np

#### Datos para la descarga (usuario, contraseña, simulación, snapnum, fragmentos de la query)

In [24]:
# Usuario y contraseña
usr='cht015'
pwd='BH457tfj'

# =========================================================================================

# Simulación y snapnum deseado
simu='RefL0100N1504'
snap=28

# =========================================================================================

# Carpeta donde guardar archivo. Dejar un caracter 'vacío' para descargar los datos
# en la misma carpeta desde donde se ejecuta la notebook.
download_folder='/home/ramiro/Documentos/Facultad/Datos guardados/Maestría/Taller de Tesis 1/Python Taller 1/data/'

# Nombre del archivo que guardará los datos descargados
filename='MuestraMZR_'+simu+'_snap_'+str(snap)+'.dat'

# =========================================================================================
# Cosas para armar la query

# Tablas y Alias de las tablas desde donde quiero descargar datos
# Agregar/quitar según sea necesario
tables=[
        'Subhalo'
          ]

aliases=[
         'sub'

        ]

# --------------------------------------------------------------------------------

# Columnas a seleccionar de cada tabla (agregar/quitar según sea necesario)
columns=[ [
           'GalaxyID','GroupID','SnapNum','SubGroupNumber','Stars_Mass','BlackHoleMass','SF_Mass','SF_Oxygen','SF_Hydrogen'
          ]   
             ]

# --------------------------------------------------------------------------------

# Condiciones sobre las columnas de las tablas a seleccionar
# NO AGREGAR ACÁ EL ALIAS DE LA TABLA! Se agregará después
# Agregar/quitar según sea necesario
# Cada lista de la siguiente lista establece condiciones para la tabla
# a seleccionar (respetando el orden en que se definieron las tablas).
# Si no se quieren condiciones sobre alguna tabla, dejar la correspondiente
# lista como vacía.

conditions=[
            ['SnapNum='+str(snap),'BlackHoleMass>'+str(0),'Stars_Mass>'+str(1e10),'SF_Oxygen>'+str(0),'SF_Hydrogen>'+str(0)]
           ]


# --------------------------------------------------------------------------------


#### Construcción de las distintas partes de la query

In [25]:
# Armo la sentencia SELECT 
select=''
for alias,col in zip(aliases,columns):
    select=select+(','.join([alias+'.'+name for name in col]))+','
select=select[:-1]   # Esto es para borrar una última coma 'molesta'

# --------------------------------------------------------------------------------

# Armo la sentencia FROM
from_table=''
for tab,alias in zip(tables,aliases):
    from_table=from_table+simu+'_'+tab+' as '+alias+','
from_table=from_table[:-1]   # Esto es para borrar una última coma 'molesta'

# --------------------------------------------------------------------------------

# Vemos las condiciones para el WHERE.
for cond in conditions:
    print(cond, type(cond))


# Armo la sentencia WHERE
# Condiciones para unir tablas (igualdad de GalaxyIDs)
join_conditions=''
# Lo sigueinte solo se ejecutará si hay más de una tabla
if len(tables)>1:
    for k in range(len(aliases)-1):
        join_conditions=(join_conditions+
                         aliases[k]+'.GalaxyID='+aliases[k+1]+'.GalaxyID'+' and ')
    join_conditions=join_conditions[:-5]

where=''    
for alias,condit in zip(aliases,conditions):
    if len(condit)>0:
        where=where+(' and '.join(alias+'.'+cond for cond in condit))+ ' and '
        if len(aliases)==1:
            where=where[:-4] # Esto es para borrar un 'and' molesto
where=where+join_conditions

['SnapNum=28', 'BlackHoleMass>0', 'Stars_Mass>10000000000.0', 'SF_Oxygen>0', 'SF_Hydrogen>0'] <class 'list'>


In [27]:
# Testeo de la query
print('SELECT')
print(select)
print()
print('FROM')
print(from_table)
print()
print('WHERE')
print(where)

SELECT
sub.GalaxyID,sub.GroupID,sub.SnapNum,sub.SubGroupNumber,sub.Stars_Mass,sub.BlackHoleMass,sub.SF_Mass,sub.SF_Oxygen,sub.SF_Hydrogen

FROM
RefL0100N1504_Subhalo as sub

WHERE
sub.SnapNum=28 and sub.BlackHoleMass>0 and sub.Stars_Mass>10000000000.0 and sub.SF_Oxygen>0 and sub.SF_Hydrogen>0 


#### Conexión a la base de datos y descarga

In [28]:
# Conectarse a la base de datos
con = sql.connect(usr,password=pwd)

# Query en SQL
query = 'SELECT TOP 10'+select+' FROM '+from_table #+' WHERE '+where

# Execute query 
exquery = sql.execute_query(con, query)

# List of column names of downloaded data
colnames=(exquery.view(np.recarray).dtype.names)

# Dictionary of data
mytable={}
for name in colnames:
    mytable[name]=exquery[name]
    
# dictionary={key1:value,key2:value,....}
    

#### Guardar los datos en un archivo ascii

In [18]:
# Abrir el archivo donde se guardarán los datos
outf = open(download_folder+filename,'w')

# Transformar el diccionario a una tabla ascii
data_ascii=Table(mytable)

# Escribir los datos al archivo
data_ascii.write(outf,format='csv')

# Cerrar el archivo
outf.close()

In [19]:
data_ascii

GalaxyID,GroupID,SnapNum,SubGroupNumber,Stars_Mass,BlackHoleMass,SF_Mass,SF_Oxygen,SF_Hydrogen
int64,int64,int32,int32,float32,float32,float32,float32,float32
8633758,28000000000698,28,1,1.1249181e+10,2.7557768e+07,2.7364926e+07,0.013665597,0.68973666
8636849,28000000000711,28,1,2.3051682e+10,9.010794e+06,3.1094874e+09,0.013035937,0.6843174
8638278,28000000000713,28,1,1.3847315e+10,1.7351778e+06,2.4194476e+09,0.013081433,0.6828084
8640505,28000000000729,28,1,2.0462213e+10,4.5367132e+07,8.19207e+08,0.014419746,0.6733895
8641614,28000000000730,28,1,1.2950536e+10,1.04480625e+06,1.3874469e+09,0.01894173,0.6455099
8643937,28000000000740,28,1,1.5268708e+10,890207.0,4.1545672e+09,0.010818631,0.69639486
8646377,28000000000765,28,0,2.3953957e+10,2.332017e+06,6.3366364e+09,0.0138172535,0.6808217
8648067,28000000000765,28,1,1.2988199e+10,2.5520372e+06,2.393118e+09,0.011800422,0.69204444
8649268,28000000000766,28,1,1.0706503e+10,3.5149038e+06,1.1845908e+09,0.014447986,0.6721208


#### EoP